# 1. Charger les données

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve
)

from xgboost import XGBClassifier

# Racine du projet
root = Path.cwd()

if root.name == "notebooks":
    root = root.parent

# Chargement des datasets
train = pd.read_parquet(root / "data/processed/train.parquet")
val = pd.read_parquet(root / "data/processed/validation.parquet")
test = pd.read_parquet(root / "data/processed/test.parquet")

# Features et cible
X_train = train.drop(columns=["Class"])
y_train = train["Class"]

X_val = val.drop(columns=["Class"])
y_val = val["Class"]

X_test = test.drop(columns=["Class"])
y_test = test["Class"]

print("Train :", X_train.shape)
print("Validation :", X_val.shape)
print("Test :", X_test.shape)

print("\nFraudes dans le train :", y_train.sum())

Train : (199364, 30)
Validation : (42721, 30)
Test : (42722, 30)

Fraudes dans le train : 384


# 2. Charger le preprocessing de l'étape 2

In [2]:
preprocessor = joblib.load(
    root / "models/preprocessor.joblib"
)

print(preprocessor)

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Time', 'Amount']),
                                ('pca',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7',
                                  'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14',
                                  'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
                                  'V21', 'V22', 'V23', 'V24', 'V25', 'V26',
                                  'V27', 'V28'])],
                  verbose_feature_names_out=False)


### Vérifier qu'il n'y a pas de NaN

In [3]:
print("Nombre total de NaN :", X_train.isna().sum().sum())

print("\nNaN par colonne :")
print(X_train.isna().sum()[X_train.isna().sum() > 0])

Nombre total de NaN : 1

NaN par colonne :
V5    1
dtype: int64


In [4]:
from sklearn.base import clone
import numpy as np

X_processed = clone(preprocessor).fit_transform(X_train)

print("NaN après preprocessing :",
      np.isnan(X_processed).sum())

NaN après preprocessing : 0


# 3. Modèle 1 — Régression logistique

In [5]:
from sklearn.base import clone

logistic_pipeline = Pipeline([
    ("preprocessing", clone(preprocessor)),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_pipeline.fit(X_train, y_train)

print("Régression logistique entraînée.")

Régression logistique entraînée.


In [6]:
proba_logistic = logistic_pipeline.predict_proba(X_val)[:, 1]

print(proba_logistic[:10])

[0.0413974  0.00550695 0.03707397 0.04468609 0.0240271  0.01133363
 0.02031448 0.03376082 0.10028411 0.00070526]


## Évaluer la régression logistique

In [7]:
def evaluate_model(y_true, probabilities, threshold=0.5):

    predictions = (
        probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_true, predictions, zero_division=0
    )

    recall = recall_score(
        y_true, predictions, zero_division=0
    )

    f1 = f1_score(
        y_true, predictions, zero_division=0
    )

    pr_auc = average_precision_score(
        y_true, probabilities
    )

    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-score  : {f1:.4f}")
    print(f"AP        : {pr_auc:.4f}")

    print("\nMatrice de confusion :")
    print(confusion_matrix(y_true, predictions))

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "average_precision": pr_auc
    }

Ici, average_precision_score calcule l'Average Precision (AP), une métrique de synthèse de la courbe précision-rappel. Elle est liée à la PR-AUC, mais n'est pas exactement identique à toutes les méthodes d'intégration de cette courbe.

In [8]:
metrics_logistic = evaluate_model(
    y_val,
    proba_logistic,
    threshold=0.5
)

Precision : 0.0528
Recall    : 0.9286
F1-score  : 0.1000
AP        : 0.8401

Matrice de confusion :
[[41733   932]
 [    4    52]]


### TN FP
### FN TP 

# 4. Modèle 2 — XGBoost

In [9]:
n_negative = (y_train == 0).sum()
n_positive = (y_train == 1).sum()

scale_pos_weight = n_negative / n_positive

print("scale_pos_weight :", scale_pos_weight)

scale_pos_weight : 518.1770833333334


In [10]:
xgb_pipeline = Pipeline([
    ("preprocessing", clone(preprocessor)),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_pipeline.fit(X_train, y_train)

print("XGBoost entraîné.")

XGBoost entraîné.


In [11]:
proba_xgb = xgb_pipeline.predict_proba(X_val)[:, 1]

metrics_xgb = evaluate_model(
    y_val,
    proba_xgb,
    threshold=0.5
)

Precision : 0.6667
Recall    : 0.8214
F1-score  : 0.7360
AP        : 0.8500

Matrice de confusion :
[[42642    23]
 [   10    46]]


# 5. Comparer les deux modèles

In [12]:
comparison = pd.DataFrame({
    "Logistic Regression": metrics_logistic,
    "XGBoost": metrics_xgb
}).T

display(comparison.round(4))

,precision,recall,f1,average_precision
Logistic Regression,0.0528,0.9286,0.100,0.8401
XGBoost,0.6667,0.8214,0.736,0.8500


# 6. Optimiser le seuil de classification

In [15]:
# Utiliser les probabilités XGBoost sur la validation
thresholds = np.arange(0.05, 0.96, 0.01)

results = []

for threshold in thresholds:

    predictions = (
        proba_xgb >= threshold
    ).astype(int)

    results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "ap": average_precision_score(
            y_val, proba_xgb
        )
    })

threshold_results = pd.DataFrame(results)

print(threshold_results.to_numpy()[-10:])
      
best_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = float(best_row["threshold"])

print("Meilleur seuil F1 :", round(best_threshold, 3))
print(best_row)

[[0.86       0.91666667 0.78571429 0.84615385 0.84997173]
 [0.87       0.95652174 0.78571429 0.8627451  0.84997173]
 [0.88       0.95652174 0.78571429 0.8627451  0.84997173]
 [0.89       0.95652174 0.78571429 0.8627451  0.84997173]
 [0.9        0.95652174 0.78571429 0.8627451  0.84997173]
 [0.91       0.95652174 0.78571429 0.8627451  0.84997173]
 [0.92       0.97777778 0.78571429 0.87128713 0.84997173]
 [0.93       0.97777778 0.78571429 0.87128713 0.84997173]
 [0.94       0.97777778 0.78571429 0.87128713 0.84997173]
 [0.95       0.97777778 0.78571429 0.87128713 0.84997173]]
Meilleur seuil F1 : 0.92
threshold    0.920000
precision    0.977778
recall       0.785714
f1           0.871287
ap           0.849972
Name: 87, dtype: float64


# 7. Évaluer le modèle final sur le test

In [21]:
# Probabilités sur le jeu de test
proba_test = xgb_pipeline.predict_proba(X_test)[:, 1]

# Évaluation finale avec le seuil choisi sur validation
final_metrics = evaluate_model(
    y_test,
    proba_test,
    threshold=best_threshold
)

print("\nRésultats finaux sur le test :")
print(final_metrics)

Precision : 0.8444
Recall    : 0.7308
F1-score  : 0.7835
AP        : 0.7594

Matrice de confusion :
[[42663     7]
 [   14    38]]

Résultats finaux sur le test :
{'precision': 0.8444444444444444, 'recall': 0.7307692307692307, 'f1': 0.7835051546391752, 'average_precision': 0.7593612315456532}


# 8. Sauvegarder le modèle et le seuil

In [22]:
model_dir = root / "models"
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    xgb_pipeline,
    model_dir / "fraud_model.joblib"
)

joblib.dump(
    {"threshold": best_threshold},
    model_dir / "decision_config.joblib"
)

print("Modèle et seuil sauvegardés.")

Modèle et seuil sauvegardés.


### Verifier :

In [23]:
loaded_model = joblib.load(
    model_dir / "fraud_model.joblib"
)

loaded_config = joblib.load(
    model_dir / "decision_config.joblib"
)

sample = X_test.iloc[:5]

probabilities = loaded_model.predict_proba(sample)[:, 1]

predictions = (
    probabilities >= loaded_config["threshold"]
).astype(int)

for p, prediction in zip(probabilities, predictions):
    print(
        f"Probabilité : {p:.4f} | "
        f"Classe : {prediction}"
    )

Probabilité : 0.0001 | Classe : 0
Probabilité : 0.0003 | Classe : 0
Probabilité : 0.0001 | Classe : 0
Probabilité : 0.0002 | Classe : 0
Probabilité : 0.0000 | Classe : 0
